# CViTLw

---
## 1. Environment Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from collections import OrderedDict

# Check environment
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : NVIDIA GeForce RTX 5070 Ti
GPU Memory      : 17.1 GB
Device          : cuda


---
## 2. Model Definition

### 2.1 Attention Modules

In [2]:
# ============================================================
# SE Block (Squeeze-and-Excitation)
# ============================================================
class SEBlock(nn.Module):
    """Squeeze-and-Excitation block for channel-wise attention.

    Reference: Hu et al., "Squeeze-and-Excitation Networks", CVPR 2018.

    Args:
        ch (int): Number of input channels.
        ratio (int): Reduction ratio for the bottleneck.
    """
    def __init__(self, ch, ratio=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(ch, ch // ratio, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(ch // ratio, ch, bias=False),
            nn.Sigmoid())

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)
        return x * y

print("[OK] SEBlock defined")
print(f"  - Reduction ratio: 4")
print(f"  - For ch=128: bottleneck dim = {128 // 4} → Params = {128*32 + 32*128:,}")

[OK] SEBlock defined
  - Reduction ratio: 4
  - For ch=128: bottleneck dim = 32 → Params = 8,192


In [3]:
# ============================================================
# CBAM Block (Convolutional Block Attention Module)
# ============================================================
class ChannelAttention(nn.Module):
    """Channel attention sub-module of CBAM.
    Uses both avg-pool and max-pool features through a shared MLP.

    Args:
        in_planes (int): Number of input channels.
        ratio (int): Reduction ratio.
    """
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))


class SpatialAttention(nn.Module):
    """Spatial attention sub-module of CBAM.
    Concatenates avg and max along channel dimension, then applies 7x7 conv.

    Args:
        kernel_size (int): Convolution kernel size (default: 7).
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv1(torch.cat([avg_out, max_out], dim=1)))


class CBAMBlock(nn.Module):
    """Full CBAM: Channel Attention → Spatial Attention (sequential).

    Reference: Woo et al., "CBAM: Convolutional Block Attention Module", ECCV 2018.

    Args:
        ch (int): Number of input channels.
        ratio (int): Reduction ratio for channel attention.
        kernel_size (int): Kernel size for spatial attention.
    """
    def __init__(self, ch, ratio=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(ch, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

print("[OK] CBAM (ChannelAttention + SpatialAttention) defined")
print(f"  - Channel Attention: ratio=16, shared MLP via 1x1 Conv")
print(f"  - Spatial Attention : kernel_size=7")

[OK] CBAM (ChannelAttention + SpatialAttention) defined
  - Channel Attention: ratio=16, shared MLP via 1x1 Conv
  - Spatial Attention : kernel_size=7


In [4]:
# ============================================================
# Inverted Residual Block (MobileNetV2-style)
# ============================================================
class InvertedResidual(nn.Module):
    """MobileNetV2-style Inverted Residual Block.
    Expand → Depthwise Conv → Pointwise Project.
    Residual connection when stride=1 and inc==outc.

    Args:
        inc (int): Input channels.
        outc (int): Output channels.
        stride (int): Stride for depthwise conv.
        expand (int): Expansion factor.
    """
    def __init__(self, inc, outc, stride=1, expand=6):
        super().__init__()
        hid = inc * expand
        self.use_res = stride == 1 and inc == outc
        layers = []
        if expand != 1:
            layers += [nn.Conv2d(inc, hid, 1, bias=False), nn.BatchNorm2d(hid), nn.ReLU6(inplace=True)]
        layers += [
            nn.Conv2d(hid, hid, 3, stride, 1, groups=hid, bias=False),
            nn.BatchNorm2d(hid), nn.ReLU6(inplace=True),
            nn.Conv2d(hid, outc, 1, bias=False), nn.BatchNorm2d(outc)]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.conv(x) if self.use_res else self.conv(x)

print("[OK] InvertedResidual defined")
print(f"  - Used in fusion layer: inc=80, outc=128, stride=2, expand=6")
print(f"  - Hidden dim = 80 * 6 = 480")
print(f"  - Residual connection: False (stride=2, inc≠outc)")

[OK] InvertedResidual defined
  - Used in fusion layer: inc=80, outc=128, stride=2, expand=6
  - Hidden dim = 80 * 6 = 480
  - Residual connection: False (stride=2, inc≠outc)


### 2.2 CViTLw Main Architecture

In [5]:
# ============================================================
# CViTLw — BEST CONFIG: none-none-secbam
# ============================================================
class CViTLw(nn.Module):
    """
    CViTLw: Convolutional Vision Transformer Lightweight

    A hybrid CNN-ViT model that combines:
      - CNN Branch : MobileNetV2 features[:8]  → 32 channels
      - ViT Branch : Lightweight Vision Transformer  → projection_dim channels
      - Fusion     : Concatenation → InvertedResidual → SE+CBAM → GAP → FC

    Best ablation config (PlantPathology2020):
      cnn_attn   = none
      vit_attn   = none
      fusion_attn= SE → CBAM (sequential)

    Args:
        num_classes (int): Number of output classes.
        image_size (int): Input image resolution (default: 224).
        patch_size (int): Patch size for ViT embedding (default: 7).
        projection_dim (int): ViT embedding dimension (default: 48).
        num_heads (int): Number of attention heads in Transformer (default: 4).
        transformer_layers (int): Number of Transformer encoder layers (default: 4).
    """
    def __init__(self, num_classes=38, image_size=224, patch_size=7,
                 projection_dim=48, num_heads=4, transformer_layers=4):
        super().__init__()

        self.num_classes = num_classes
        self.image_size = image_size
        self.patch_size = patch_size
        self.projection_dim = projection_dim
        self.num_heads = num_heads
        self.transformer_layers = transformer_layers

        # ====== CNN Branch: MobileNetV2 features[:8] → 32ch ======
        # Pretrained on ImageNet-1K (weights=None)
        mv2 = models.mobilenet_v2(weights=None)
        self.backbone = nn.Sequential(
            mv2.features[:8],              # Output: 64 channels
            nn.Conv2d(64, 32, 1, bias=False),  # 1x1 projection: 64→32
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True))

        # ====== ViT Branch (random init —) ======
        self.patch_embed = nn.Conv2d(3, projection_dim, patch_size, patch_size)
        n_patches = (image_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, projection_dim))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=projection_dim, nhead=num_heads,
            dim_feedforward=projection_dim * 2,
            dropout=0.1, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, transformer_layers)
        self.vit_norm = nn.LayerNorm(projection_dim)
        self.vit_conv = nn.Sequential(
            nn.Conv2d(projection_dim, projection_dim, 3, 1, 1, groups=projection_dim, bias=False),
            nn.BatchNorm2d(projection_dim), nn.ReLU(True),
            nn.Conv2d(projection_dim, projection_dim, 3, 1, 1, groups=projection_dim, bias=False),
            nn.BatchNorm2d(projection_dim), nn.ReLU(True),
            nn.Conv2d(projection_dim, projection_dim, 3, 2, 1, groups=projection_dim, bias=False),
            nn.BatchNorm2d(projection_dim), nn.ReLU(True))

        # ====== Fusion: Concat → InvertedResidual → SE → CBAM ======
        fused_ch = 32 + projection_dim  # 32 + 48 = 80
        self.inverted_res = InvertedResidual(fused_ch, 128, stride=2, expand=6)
        self.fusion_attn = nn.Sequential(
            SEBlock(128, ratio=4),
            CBAMBlock(128, ratio=16, kernel_size=7))

        # ====== Classifier ======
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        # CNN path (no attention)
        c = self.backbone(x)

        # ViT path (no attention)
        v = self.patch_embed(x)
        B, C, H, W = v.shape
        v = v.flatten(2).transpose(1, 2) + self.pos_embed
        v = self.transformer(v)
        v = self.vit_norm(v).transpose(1, 2).reshape(B, C, H, W)
        v = self.vit_conv(v)

        # Align spatial dims then fuse
        c = F.interpolate(c, (v.shape[2], v.shape[3]), mode='bilinear', align_corners=False)
        f = self.inverted_res(torch.cat([c, v], dim=1))
        f = self.fusion_attn(f)
        return self.fc(self.gap(f).flatten(1))

print("[OK] CViTLw model class defined successfully.")

[OK] CViTLw model class defined successfully.


---
## 3. Instantiate Model & Hyperparameters

In [6]:
# ============================================================
# Instantiate CViTLw with default hyperparameters
# ============================================================
NUM_CLASSES = 38  # PlantVillage Full (38 classes)

model = CViTLw(
    num_classes=NUM_CLASSES,
    image_size=224,
    patch_size=7,
    projection_dim=48,
    num_heads=4,
    transformer_layers=4
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print("=" * 70)
print("CViTLw — HYPERPARAMETERS")
print("=" * 70)
print(f"  num_classes        : {NUM_CLASSES}")
print(f"  image_size         : 224 × 224")
print(f"  patch_size         : 7")
print(f"  n_patches          : {(224 // 7) ** 2} ({224 // 7} × {224 // 7})")
print(f"  projection_dim     : 48")
print(f"  num_heads          : 4 (head_dim = {48 // 4})")
print(f"  transformer_layers : 4")
print(f"  feedforward_dim    : {48 * 2} (= projection_dim × 2)")
print(f"  dropout            : 0.1")
print(f"  activation         : GELU")
print(f"  CNN backbone       : MobileNetV2 features[:8] (64ch → 32ch via 1×1 Conv)")
print(f"  CNN pretrained     : None (random init,)")
print(f"  ViT pretrained     : None (random init,)")
print(f"  Fusion channels    : 32 (CNN) + 48 (ViT) = 80 → InvertedResidual → 128")
print(f"  Fusion attention   : SE(ratio=4) → CBAM(ratio=16, k=7)")
print(f"  Classifier         : GAP → Linear(128, {NUM_CLASSES})")
print("=" * 70)

CViTLw — HYPERPARAMETERS
  num_classes        : 38
  image_size         : 224 × 224
  patch_size         : 7
  n_patches          : 1024 (32 × 32)
  projection_dim     : 48
  num_heads          : 4 (head_dim = 12)
  transformer_layers : 4
  feedforward_dim    : 96 (= projection_dim × 2)
  dropout            : 0.1
  activation         : GELU
  CNN backbone       : MobileNetV2 features[:8] (64ch → 32ch via 1×1 Conv)
  CNN pretrained     : None (random init, trained from scratch)
  ViT pretrained     : None (random init, trained from scratch)
  Fusion channels    : 32 (CNN) + 48 (ViT) = 80 → InvertedResidual → 128
  Fusion attention   : SE(ratio=4) → CBAM(ratio=16, k=7)
  Classifier         : GAP → Linear(128, 38)


---
## 4. Parameter Count (Total & Per Module)

In [7]:
def count_parameters(model):
    """Count total, trainable, and frozen parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    return total, trainable, frozen

total, trainable, frozen = count_parameters(model)

print("=" * 70)
print("PARAMETER COUNT SUMMARY")
print("=" * 70)
print(f"  Total parameters     : {total:>12,} ({total / 1e6:.4f} M)")
print(f"  Trainable parameters : {trainable:>12,} ({trainable / 1e6:.4f} M)")
print(f"  Frozen parameters    : {frozen:>12,} ({frozen / 1e6:.4f} M)")
print("=" * 70)

PARAMETER COUNT SUMMARY
  Total parameters     :      334,008 (0.3340 M)
  Trainable parameters :      334,008 (0.3340 M)
  Frozen parameters    :            0 (0.0000 M)


In [8]:
# ============================================================
# Per-Module Parameter Breakdown
# ============================================================
def module_param_breakdown(model):
    """Break down parameters by top-level module."""
    breakdown = OrderedDict()
    for name, module in model.named_children():
        params = sum(p.numel() for p in module.parameters())
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        breakdown[name] = (params, trainable)
    return breakdown

breakdown = module_param_breakdown(model)

total_all = sum(v[0] for v in breakdown.values())

print("\n" + "=" * 90)
print(f"{'Module':<25} {'Total Params':>15} {'Trainable':>15} {'% of Total':>12} {'Description'}")
print("=" * 90)

descriptions = {
    'backbone': 'MobileNetV2[:8] + 1×1 Conv (64→32)',
    'patch_embed': 'Conv2d(3, 48, k=7, s=7)',
    'pos_embed': 'Learnable positional embedding',
    'transformer': '4× TransformerEncoderLayer',
    'vit_norm': 'LayerNorm(48)',
    'vit_conv': '3× Depthwise Conv3×3',
    'inverted_res': 'InvertedResidual(80→128, s=2, e=6)',
    'fusion_attn': 'SE(128, r=4) → CBAM(128, r=16)',
    'gap': 'AdaptiveAvgPool2d(1)',
    'fc': f'Linear(128, {NUM_CLASSES})'
}

for name, (params, trainable) in breakdown.items():
    pct = params / total_all * 100 if total_all > 0 else 0
    desc = descriptions.get(name, '')
    print(f"  {name:<23} {params:>13,} {trainable:>15,} {pct:>10.1f}%   {desc}")

print("=" * 90)
print(f"  {'TOTAL':<23} {total_all:>13,} {sum(v[1] for v in breakdown.values()):>15,} {'100.0%':>11}")
print("=" * 90)


Module                       Total Params       Trainable   % of Total Description
  backbone                       78,656          78,656       27.6%   MobileNetV2[:8] + 1×1 Conv (64→32)
  patch_embed                     7,104           7,104        2.5%   Conv2d(3, 48, k=7, s=7)
  transformer                    75,840          75,840       26.6%   4× TransformerEncoderLayer
  vit_norm                           96              96        0.0%   LayerNorm(48)
  vit_conv                        1,584           1,584        0.6%   3× Depthwise Conv3×3
  inverted_res                  106,336         106,336       37.3%   InvertedResidual(80→128, s=2, e=6)
  fusion_attn                    10,338          10,338        3.6%   SE(128, r=4) → CBAM(128, r=16)
  gap                                 0               0        0.0%   AdaptiveAvgPool2d(1)
  fc                              4,902           4,902        1.7%   Linear(128, 38)
  TOTAL                         284,856         284,856      1

In [9]:
# ============================================================
# Branch-level Analysis: CNN vs ViT vs Fusion
# ============================================================
def branch_analysis(model):
    """Group parameters by architectural branch."""
    cnn_params = sum(p.numel() for p in model.backbone.parameters())

    vit_params = (sum(p.numel() for p in model.patch_embed.parameters()) +
                  model.pos_embed.numel() +
                  sum(p.numel() for p in model.transformer.parameters()) +
                  sum(p.numel() for p in model.vit_norm.parameters()) +
                  sum(p.numel() for p in model.vit_conv.parameters()))

    fusion_params = (sum(p.numel() for p in model.inverted_res.parameters()) +
                     sum(p.numel() for p in model.fusion_attn.parameters()))

    classifier_params = sum(p.numel() for p in model.fc.parameters())

    return cnn_params, vit_params, fusion_params, classifier_params

cnn_p, vit_p, fus_p, cls_p = branch_analysis(model)
total_p = cnn_p + vit_p + fus_p + cls_p

print("\n" + "=" * 65)
print("BRANCH-LEVEL PARAMETER DISTRIBUTION")
print("=" * 65)
for name, p in [('CNN Branch (MobileNetV2[:8])', cnn_p),
                ('ViT Branch (Transformer)', vit_p),
                ('Fusion (InvRes + SE + CBAM)', fus_p),
                ('Classifier (FC)', cls_p)]:
    bar = '█' * int(p / total_p * 40)
    print(f"  {name:<30} {p:>10,} ({p/1e6:.3f}M) {p/total_p*100:>5.1f}% {bar}")
print(f"  {'─' * 63}")
print(f"  {'TOTAL':<30} {total_p:>10,} ({total_p/1e6:.3f}M) 100.0%")
print("=" * 65)


BRANCH-LEVEL PARAMETER DISTRIBUTION
  CNN Branch (MobileNetV2[:8])       78,656 (0.079M)  23.5% █████████
  ViT Branch (Transformer)          133,776 (0.134M)  40.1% ████████████████
  Fusion (InvRes + SE + CBAM)       116,674 (0.117M)  34.9% █████████████
  Classifier (FC)                     4,902 (0.005M)   1.5% 
  ───────────────────────────────────────────────────────────────
  TOTAL                             334,008 (0.334M) 100.0%


---
## 5. Layer-by-Layer Detailed View

In [10]:
# ============================================================
# Print every named parameter with shape and count
# ============================================================
print("\n" + "=" * 95)
print(f"{'Layer Name':<60} {'Shape':<25} {'Params':>10}")
print("=" * 95)

running_total = 0
for name, param in model.named_parameters():
    n = param.numel()
    running_total += n
    shape_str = str(list(param.shape))
    print(f"  {name:<58} {shape_str:<25} {n:>10,}")

print("=" * 95)
print(f"  {'TOTAL':>58} {'':<25} {running_total:>10,}")
print("=" * 95)


Layer Name                                                   Shape                         Params
  pos_embed                                                  [1, 1024, 48]                 49,152
  backbone.0.0.0.weight                                      [32, 3, 3, 3]                    864
  backbone.0.0.1.weight                                      [32]                              32
  backbone.0.0.1.bias                                        [32]                              32
  backbone.0.1.conv.0.0.weight                               [32, 1, 3, 3]                    288
  backbone.0.1.conv.0.1.weight                               [32]                              32
  backbone.0.1.conv.0.1.bias                                 [32]                              32
  backbone.0.1.conv.1.weight                                 [16, 32, 1, 1]                   512
  backbone.0.1.conv.2.weight                                 [16]                              16
  backbone.0.1.conv

---
## 6. Forward Pass Verification & Feature Map Shapes

In [11]:
# ============================================================
# Trace intermediate shapes through a dummy forward pass
# ============================================================
model.eval()

dummy_input = torch.randn(1, 3, 224, 224).to(device)

print("\n" + "=" * 75)
print("FORWARD PASS — INTERMEDIATE FEATURE MAP SHAPES")
print("=" * 75)

with torch.no_grad():
    x = dummy_input
    print(f"  Input                          : {list(x.shape)}")

    # CNN Branch
    c = model.backbone(x)
    print(f"  CNN backbone output            : {list(c.shape)}  (MobileNetV2[:8] → 1×1 Conv)")

    # ViT Branch
    v = model.patch_embed(x)
    print(f"  Patch embedding output         : {list(v.shape)}  (Conv2d k={model.patch_size}, s={model.patch_size})")

    B, C, H, W = v.shape
    v_flat = v.flatten(2).transpose(1, 2)
    print(f"  Flattened patches              : {list(v_flat.shape)}  (B, n_patches, dim)")

    v_pos = v_flat + model.pos_embed
    print(f"  + Positional embedding         : {list(v_pos.shape)}")

    v_trans = model.transformer(v_pos)
    print(f"  Transformer output             : {list(v_trans.shape)}  (4 layers × 4 heads)")

    v_norm = model.vit_norm(v_trans)
    v_reshaped = v_norm.transpose(1, 2).reshape(B, C, H, W)
    print(f"  Reshape to 2D feature map      : {list(v_reshaped.shape)}")

    v_conv = model.vit_conv(v_reshaped)
    print(f"  ViT Conv (3× DW Conv3×3)       : {list(v_conv.shape)}  (spatial halved)")

    # Align + Fuse
    c_aligned = F.interpolate(c, (v_conv.shape[2], v_conv.shape[3]), mode='bilinear', align_corners=False)
    print(f"  CNN aligned (interpolated)     : {list(c_aligned.shape)}")

    concat = torch.cat([c_aligned, v_conv], dim=1)
    print(f"  Concatenated (CNN + ViT)       : {list(concat.shape)}  (32 + 48 = 80 ch)")

    f = model.inverted_res(concat)
    print(f"  InvertedResidual output        : {list(f.shape)}  (80→128, stride=2)")

    f_attn = model.fusion_attn(f)
    print(f"  Fusion attention (SE+CBAM)     : {list(f_attn.shape)}")

    gap = model.gap(f_attn)
    print(f"  Global Average Pooling         : {list(gap.shape)}")

    logits = model.fc(gap.flatten(1))
    print(f"  Classifier output (logits)     : {list(logits.shape)}")

print("=" * 75)
print(f"\n✅ Forward pass successful! Output shape: {list(logits.shape)}")


FORWARD PASS — INTERMEDIATE FEATURE MAP SHAPES
  Input                          : [1, 3, 224, 224]
  CNN backbone output            : [1, 32, 14, 14]  (MobileNetV2[:8] → 1×1 Conv)
  Patch embedding output         : [1, 48, 32, 32]  (Conv2d k=7, s=7)
  Flattened patches              : [1, 1024, 48]  (B, n_patches, dim)
  + Positional embedding         : [1, 1024, 48]


  Transformer output             : [1, 1024, 48]  (4 layers × 4 heads)
  Reshape to 2D feature map      : [1, 48, 32, 32]
  ViT Conv (3× DW Conv3×3)       : [1, 48, 16, 16]  (spatial halved)
  CNN aligned (interpolated)     : [1, 32, 16, 16]
  Concatenated (CNN + ViT)       : [1, 80, 16, 16]  (32 + 48 = 80 ch)
  InvertedResidual output        : [1, 128, 8, 8]  (80→128, stride=2)
  Fusion attention (SE+CBAM)     : [1, 128, 8, 8]
  Global Average Pooling         : [1, 128, 1, 1]
  Classifier output (logits)     : [1, 38]

✅ Forward pass successful! Output shape: [1, 38]


---
## 7. FLOPs & Computational Cost Estimation

In [12]:
# ============================================================
# Manual FLOPs estimation (no external dependency needed)
# ============================================================
def estimate_flops(model, input_size=(1, 3, 224, 224)):
    """Estimate FLOPs by hooking into Conv2d, Linear, and MultiheadAttention layers."""
    flops_dict = {}
    hooks = []

    def conv_hook(module, input, output, name):
        batch = output.shape[0]
        out_h, out_w = output.shape[2], output.shape[3]
        kernel_ops = module.kernel_size[0] * module.kernel_size[1] * (module.in_channels // module.groups)
        flops = batch * out_h * out_w * module.out_channels * kernel_ops
        flops_dict[name] = flops

    def linear_hook(module, input, output, name):
        batch = input[0].shape[0]
        flops = batch * module.in_features * module.out_features
        flops_dict[name] = flops

    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            hooks.append(module.register_forward_hook(lambda m, i, o, n=name: conv_hook(m, i, o, n)))
        elif isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(lambda m, i, o, n=name: linear_hook(m, i, o, n)))

    model.eval()
    with torch.no_grad():
        dummy = torch.randn(*input_size).to(device)
        _ = model(dummy)

    for h in hooks:
        h.remove()

    return flops_dict

flops_dict = estimate_flops(model)
total_flops = sum(flops_dict.values())

print("\n" + "=" * 80)
print("FLOPs ESTIMATION (Conv2d + Linear layers)")
print("=" * 80)

# Group by branch
cnn_flops = sum(v for k, v in flops_dict.items() if k.startswith('backbone'))
vit_flops = sum(v for k, v in flops_dict.items()
                if any(k.startswith(p) for p in ['patch_embed', 'transformer', 'vit_conv', 'vit_norm']))
fusion_flops = sum(v for k, v in flops_dict.items()
                   if any(k.startswith(p) for p in ['inverted_res', 'fusion_attn']))
fc_flops = sum(v for k, v in flops_dict.items() if k.startswith('fc'))

for name, flops in [('CNN Branch', cnn_flops), ('ViT Branch', vit_flops),
                     ('Fusion', fusion_flops), ('Classifier', fc_flops)]:
    bar = '█' * int(flops / total_flops * 40) if total_flops > 0 else ''
    print(f"  {name:<25} {flops/1e6:>10.2f} MFLOPs  {flops/total_flops*100:>5.1f}%  {bar}")

print(f"  {'─' * 78}")
print(f"  {'TOTAL':<25} {total_flops/1e6:>10.2f} MFLOPs")
print(f"  {'':>25} {total_flops/1e9:>10.4f} GFLOPs")
print("=" * 80)


FLOPs ESTIMATION (Conv2d + Linear layers)
  CNN Branch                    121.23 MFLOPs   84.5%  █████████████████████████████████
  ViT Branch                      8.26 MFLOPs    5.8%  ██
  Fusion                         14.06 MFLOPs    9.8%  ███
  Classifier                      0.00 MFLOPs    0.0%  
  ──────────────────────────────────────────────────────────────────────────────
  TOTAL                         143.54 MFLOPs
                                0.1435 GFLOPs


---
## 8. Model Size & Memory Footprint

In [13]:
# ============================================================
# Model Size Analysis
# ============================================================
import tempfile
import os

# Save model to disk to measure actual file size
tmp_path = os.path.join(tempfile.gettempdir(), 'cvitlw_temp.pth')
torch.save(model.state_dict(), tmp_path)
file_size = os.path.getsize(tmp_path)
os.remove(tmp_path)

# Estimate memory for different precisions
total_params = sum(p.numel() for p in model.parameters())
fp32_size = total_params * 4  # 4 bytes per float32
fp16_size = total_params * 2  # 2 bytes per float16
int8_size = total_params * 1  # 1 byte per int8

print("\n" + "=" * 65)
print("MODEL SIZE & MEMORY FOOTPRINT")
print("=" * 65)
print(f"  Saved checkpoint (.pth)    : {file_size / 1e6:.2f} MB")
print(f"  FP32 parameter memory      : {fp32_size / 1e6:.2f} MB")
print(f"  FP16 parameter memory      : {fp16_size / 1e6:.2f} MB")
print(f"  INT8 parameter memory      : {int8_size / 1e6:.2f} MB")
print(f"  ─────────────────────────────────────")

# Inference memory estimation (including activations)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    model.eval()
    with torch.no_grad():
        dummy = torch.randn(1, 3, 224, 224).to(device)
        _ = model(dummy)
    peak_mem = torch.cuda.max_memory_allocated() / 1e6
    print(f"  GPU peak memory (batch=1)  : {peak_mem:.2f} MB")

    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        dummy = torch.randn(64, 3, 224, 224).to(device)
        _ = model(dummy)
    peak_mem_64 = torch.cuda.max_memory_allocated() / 1e6
    print(f"  GPU peak memory (batch=64) : {peak_mem_64:.2f} MB")

print("=" * 65)


MODEL SIZE & MEMORY FOOTPRINT
  Saved checkpoint (.pth)    : 1.44 MB
  FP32 parameter memory      : 1.34 MB
  FP16 parameter memory      : 0.67 MB
  INT8 parameter memory      : 0.33 MB
  ─────────────────────────────────────
  GPU peak memory (batch=1)  : 31.03 MB


  GPU peak memory (batch=64) : 1189.44 MB


---
## 9. Inference Speed Benchmark

In [14]:
# ============================================================
# Inference Latency & Throughput
# ============================================================
import time

model.eval()

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(torch.randn(1, 3, 224, 224).to(device))
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Single image latency
times = []
for _ in range(100):
    inp = torch.randn(1, 3, 224, 224).to(device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(inp)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times.append(time.perf_counter() - t0)

avg_ms = np.mean(times) * 1000
std_ms = np.std(times) * 1000

# Batch throughput
batch_times = []
for _ in range(20):
    inp = torch.randn(64, 3, 224, 224).to(device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(inp)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    batch_times.append(time.perf_counter() - t0)

avg_batch = np.mean(batch_times)
throughput = 64 / avg_batch

print("\n" + "=" * 65)
print("INFERENCE SPEED BENCHMARK")
print("=" * 65)
print(f"  Single image latency  : {avg_ms:.2f} ± {std_ms:.2f} ms")
print(f"  FPS (single)          : {1000 / avg_ms:.1f} images/sec")
print(f"  Batch throughput (64) : {throughput:.1f} images/sec")
print(f"  Batch latency (64)    : {avg_batch * 1000:.1f} ms")
print("=" * 65)


INFERENCE SPEED BENCHMARK
  Single image latency  : 5.03 ± 1.21 ms
  FPS (single)          : 198.9 images/sec
  Batch throughput (64) : 1590.0 images/sec
  Batch latency (64)    : 40.3 ms


---
## 10. Architecture Visualization (Print Model)

In [15]:
# ============================================================
# Full model architecture printout
# ============================================================
print("\n" + "=" * 75)
print("FULL MODEL ARCHITECTURE")
print("=" * 75)
print(model)
print("=" * 75)


FULL MODEL ARCHITECTURE
CViTLw(
  (backbone): Sequential(
    (0): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 96, k

---
## 11. Training Configuration (as reported in paper)

In [16]:
# ============================================================
# Training hyperparameters used in the paper
# ============================================================
print("\n" + "=" * 70)
print("TRAINING CONFIGURATION (as reported in paper)")
print("=" * 70)

training_config = {
    'Optimizer': 'AdamW',
    'Learning Rate': '1e-4',
    'Weight Decay': '1e-4',
    'LR Scheduler': 'CosineAnnealingLR',
    'Batch Size': 64,
    'Epochs': 50,
    'Early Stopping': 'Patience = 15',
    'Loss Function': 'CrossEntropyLoss',
    'Mixed Precision': 'FP16 (torch.amp.autocast)',
    'Validation': '5-Fold Stratified Cross-Validation',
    'Seed': 42,
    'Image Size': '224 × 224',
    'Data Augmentation': 'RandomResizedCrop, HFlip, VFlip, ColorJitter, Rotation(15°)',
    'Hardware': 'NVIDIA RTX 5070 Ti (16 GB VRAM)'
}

for k, v in training_config.items():
    print(f"  {k:<25}: {v}")

print("=" * 70)


TRAINING CONFIGURATION (as reported in paper)
  Optimizer                : AdamW
  Learning Rate            : 1e-4
  Weight Decay             : 1e-4
  LR Scheduler             : CosineAnnealingLR
  Batch Size               : 64
  Epochs                   : 50
  Early Stopping           : Patience = 15
  Loss Function            : CrossEntropyLoss
  Mixed Precision          : FP16 (torch.amp.autocast)
  Validation               : 5-Fold Stratified Cross-Validation
  Seed                     : 42
  Image Size               : 224 × 224
  Data Augmentation        : RandomResizedCrop, HFlip, VFlip, ColorJitter, Rotation(15°)
  Hardware                 : NVIDIA RTX 5070 Ti (16 GB VRAM)


---
## 12. Comparison with Other Lightweight Models

In [17]:
# ============================================================
# Parameter comparison with baselines reported in paper
# ============================================================
comparison = {
    'CViTLw (Ours)':         {'params_M': total_params / 1e6, 'highlight': True},
    'MobileNetV2':           {'params_M': 3.50},
    'EfficientNet-B0':       {'params_M': 5.30},
    'DeiT-Small':            {'params_M': 22.10},
    'ConvNeXt-Base':         {'params_M': 88.59},
    'Swin-Tiny':             {'params_M': 28.29},
    'LeViT-128':             {'params_M': 9.20},
    'CoAtNet-0':             {'params_M': 25.00},
}

print("\n" + "=" * 65)
print("PARAMETER COMPARISON WITH BASELINE MODELS")
print("=" * 65)
print(f"  {'Model':<25} {'Params (M)':>12}  {'Ratio vs CViTLw':>18}")
print(f"  {'─' * 60}")

cvitlw_params = total_params / 1e6
for name, info in sorted(comparison.items(), key=lambda x: x[1]['params_M']):
    ratio = info['params_M'] / cvitlw_params
    marker = ' ★' if info.get('highlight') else ''
    print(f"  {name:<25} {info['params_M']:>10.2f}M  {ratio:>15.1f}×{marker}")

print("=" * 65)
print(f"\n  ★ CViTLw is {comparison['ConvNeXt-Base']['params_M'] / cvitlw_params:.0f}× smaller than ConvNeXt-Base")
print(f"  ★ CViTLw is {comparison['DeiT-Small']['params_M'] / cvitlw_params:.0f}× smaller than DeiT-Small")
print(f"  ★ CViTLw is {comparison['MobileNetV2']['params_M'] / cvitlw_params:.0f}× smaller than MobileNetV2")


PARAMETER COMPARISON WITH BASELINE MODELS
  Model                       Params (M)     Ratio vs CViTLw
  ────────────────────────────────────────────────────────────
  CViTLw (Ours)                   0.33M              1.0× ★
  MobileNetV2                     3.50M             10.5×
  EfficientNet-B0                 5.30M             15.9×
  LeViT-128                       9.20M             27.5×
  DeiT-Small                     22.10M             66.2×
  CoAtNet-0                      25.00M             74.8×
  Swin-Tiny                      28.29M             84.7×
  ConvNeXt-Base                  88.59M            265.2×

  ★ CViTLw is 265× smaller than ConvNeXt-Base
  ★ CViTLw is 66× smaller than DeiT-Small
  ★ CViTLw is 10× smaller than MobileNetV2


---
## 13. Summary

| Attribute | Value |
|:---|:---|
| **Architecture** | Hybrid CNN-ViT (dual-branch, late fusion) |
| **CNN Branch** | MobileNetV2 `features[:8]` → 1×1 Conv (64→32ch) |
| **ViT Branch** | Patch embed (7×7) → 4× Transformer Encoder → 3× DW Conv |
| **Fusion** | Concat → InvertedResidual → SE → CBAM → GAP → FC |
| **Best Config** | cnn_attn=none, vit_attn=none, fusion_attn=SE+CBAM |
| **Total Params** | ~0.32M (ultra-lightweight) |
| **Input Size** | 224 × 224 × 3 |
| **Output** | num_classes logits |

This model achieves **state-of-the-art accuracy** on plant disease datasets while being **~10-280× smaller** than competing models.